In [ ]:
import numpy as np
import librosa
from scipy.io import wavfile
from sklearn.preprocessing import MinMaxScaler
import scipy.fftpack
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
# 전처리 및 특징 추출 파라미터
sampling_rate = 16000  # 샘플링 속도
fft_size = 1024  # FFT 사이즈
window_length = int(0.025 * sampling_rate)  # 윈도우 길이 (25ms)
hop_length = int(0.010 * sampling_rate)  # 홉 길이 (10ms)
n_mfcc = 100

In [ ]:
def load_file(filename, sampling_rate=16000):
    data, sr = librosa.load(filename, sr=sampling_rate)
    return data, sr

In [ ]:
def remove_silence(y, sr, top_db=20):
    # 무음 구간의 시작과 끝을 찾기
    intervals = librosa.effects.split(y, top_db=top_db)

    # 유효한 오디오 신호를 합치기
    y_trimmed = np.concatenate([y[start:end] for start, end in intervals])

    return y_trimmed

In [ ]:
def normalize_spectrogram(mel_spectrogram):
    scaler = MinMaxScaler()
    mel_spectrogram_norm = scaler.fit_transform(mel_spectrogram.T).T
    return mel_spectrogram_norm

In [ ]:
def compute_log_mel_spectrogram(mel_spectrogram_norm):
    return np.log(mel_spectrogram_norm + 1e-6)

In [ ]:
def apply_dct(log_mel_spectrogram, n_mfcc):
    return scipy.fftpack.dct(log_mel_spectrogram, type=2, axis=1, norm='ortho')[:, :n_mfcc]

In [ ]:
def plot_mfcc(mfcc, sampling_rate, hop_length, vmin=1, vmax=-1):
    plt.figure(figsize=(10, 6))
    plt.imshow(mfcc.T, aspect='auto', origin='lower', 
               extent=[0, mfcc.shape[0] * hop_length / sampling_rate, 0, mfcc.shape[1]], vmin=vmin, vmax=vmax)
    plt.colorbar()
    plt.title('MFCC')
    plt.xlabel('Time (s)')
    plt.ylabel('MFCC Coefficients')
    plt.tight_layout()
    plt.show()

In [ ]:
def compute_mfcc(filename, sampling_rate=16000, fft_size=1024, window_length=400, hop_length=160, n_mfcc=13):
    data, sr = load_file(filename, sampling_rate)

    # 무음 제거
    data = remove_silence(data, sr)
    
    mel_spectrogram = mel_spectrogram_generator(data, sr, fft_size, hop_length)
    mel_spectrogram_norm = normalize_spectrogram(mel_spectrogram)
    log_mel_spectrogram = compute_log_mel_spectrogram(mel_spectrogram_norm)
    mfcc = apply_dct(log_mel_spectrogram, n_mfcc)
    
    return mfcc

In [ ]:
df = pd.read_csv('../data/train.csv')
file_paths = df['path'].values
labels = df['label'].apply(lambda x: [1, 0] if x == 'real' else [0, 1]).values
# labels = np.array([np.array(label) for label in labels])

In [ ]:
real_mfcc_features = []
fake_mfcc_features = []

# MFCC 특징 추출 및 분류
for path, label in tqdm(zip(file_paths, labels), total=len(labels)):
    mfcc = compute_mfcc(path, sampling_rate, fft_size, window_length, hop_length, n_mfcc)
    if label == [1, 0]:  # 진짜 음성
        real_mfcc_features.append(mfcc)
    else:  # 가짜 음성
        fake_mfcc_features.append(mfcc)


In [ ]:
max_length = max(mfcc.shape[0] for mfcc in real_mfcc_features)
max_length

In [ ]:
max_length = max(mfcc.shape[1] for mfcc in real_mfcc_features)
max_length

In [ ]:
def pad_features(features, max_length, max_features):
    padded_features = []
    for mfcc in tqdm(features):
        # 패딩해야 할 길이를 계산
        pad_width = max_length - mfcc.shape[0]
        # 특징 수에 대한 패딩 길이를 계산 (mfcc의 열이 max_features보다 작을 경우에만)
        pad_feature_width = max_features - mfcc.shape[1] if mfcc.shape[1] < max_features else 0
        
        # pad_width나 pad_feature_width가 음수일 경우를 처리
        pad_width = max(0, pad_width)
        pad_feature_width = max(0, pad_feature_width)

        # mfcc 배열을 0으로 패딩
        mfcc = np.pad(mfcc, ((0, pad_width), (0, pad_feature_width)), mode='constant')
        
        # 패딩된 mfcc 배열을 리스트에 추가
        padded_features.append(mfcc)
    return np.array(padded_features)

In [ ]:
real_mfcc_features = pad_features(real_mfcc_features, max_length, max_features)
fake_mfcc_features = pad_features(fake_mfcc_features, max_length, max_features)

In [ ]:
np.save('real_mfcc_features.npy', real_mfcc_features)
np.save('fake_mfcc_features.npy', fake_mfcc_features)

In [ ]:
unlabeled_dir = 'unlabeled_data'
unlabeled_mfcc_features = []

In [ ]:
for file in tqdm(os.listdir(unlabeled_dir)):
    if file.endswith('.ogg'):
        path = os.path.join(unlabeled_dir, file)
        mfcc = compute_mfcc(path, sampling_rate, fft_size, window_length, hop_length, n_mfcc)
        unlabeled_mfcc_features.append(mfcc)

In [ ]:
max_unlabeled_length = max([f.shape[0] for f in unlabeled_mfcc_features])

# 패딩 적용
unlabeled_mfcc_features = pad_features(unlabeled_mfcc_features, max_unlabeled_length, max_features)

In [ ]:
np.save('unlabeled_mfcc_features.npy', unlabeled_mfcc_features)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, real_data, fake_data, unlabeled_data):
        self.real_data = real_data
        self.fake_data = fake_data
        self.unlabeled_data = unlabeled_data
        self.labeled_data = np.concatenate((real_data, fake_data), axis=0)
        self.labels = np.concatenate((np.tile([0, 1], (len(real_data), 1)), np.tile([1, 0], (len(fake_data), 1))), axis=0)
    
    def __len__(self):
        return len(self.labeled_data)
    
    def __getitem__(self, idx):
        sample = torch.tensor(self.labeled_data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return sample, label

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 12800),  # 크기에 맞게 변경
            nn.Tanh()
        )
    
    def forward(self, z):
        x = self.model(z)
        x = x.view(-1, 12800)  # 크기에 맞게 변경
        return x

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(12800, 512),  # 크기에 맞게 변경
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2),  # 두 개의 출력
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [ ]:
class Discriminator1(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1300, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 2),  # 두 개의 출력
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [ ]:
from sklearn.metrics import roc_auc_score
def train_gan(generator, discriminator, train_loader, val_loader, unlabeled_data, epochs=10000, batch_size=64, latent_dim=100, lr=0.0002):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    generator.to(device)
    discriminator.to(device)
    
    if torch.cuda.device_count() > 1:
        generator = nn.DataParallel(generator)
        discriminator = nn.DataParallel(discriminator)
    
    criterion = nn.BCELoss()
    optimizer_G = optim.Adam(generator.parameters(), lr=lr)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=lr)
    
    for epoch in range(epochs):
        generator.train()
        discriminator.train()
        train_d_loss = 0
        train_g_loss = 0
        all_real_labels = []
        all_real_outputs = []
        all_fake_labels = []
        all_fake_outputs = []
        
        for i, (real_samples, labels) in tqdm(enumerate(train_loader)):
            real_samples, labels = real_samples.to(device), labels.to(device)
            batch_size = real_samples.size(0)
            
            # 진짜 데이터로 학습
            real_labels = torch.ones(batch_size, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1).to(device)

            optimizer_D.zero_grad()
            outputs = discriminator(real_samples)
            d_loss_real = criterion(outputs[:, 1].view(-1, 1), real_labels)  # 첫 번째 출력이 진짜 여부
            d_loss_real.backward()
            
            # 가짜 데이터 생성 및 학습
            noise = torch.randn(batch_size, latent_dim).to(device)
            fake_samples = generator(noise)
            outputs = discriminator(fake_samples.detach())
            d_loss_fake = criterion(outputs[:, 0].view(-1, 1), fake_labels)  # 두 번째 출력이 가짜 여부
            d_loss_fake.backward()
            optimizer_D.step()
            
            d_loss = d_loss_real + d_loss_fake
            train_d_loss += d_loss.item()
            
            # AUC 계산을 위한 데이터 저장
            all_real_labels.extend(real_labels.cpu().numpy())
            all_real_outputs.extend(outputs[:, 1].view(-1, 1).cpu().detach().numpy())
            all_fake_labels.extend(fake_labels.cpu().numpy())
            all_fake_outputs.extend(outputs[:, 0].view(-1, 1).cpu().detach().numpy())

            # Generator 학습
            optimizer_G.zero_grad()
            outputs = discriminator(fake_samples)
            g_loss = criterion(outputs[:, 1].view(-1, 1), real_labels)  # 생성자는 진짜로 보이도록 학습
            g_loss.backward()
            optimizer_G.step()
            
            train_g_loss += g_loss.item()

        # 평균 손실 계산
        train_d_loss /= len(train_loader)
        train_g_loss /= len(train_loader)
        
        generator.eval()
        discriminator.eval()
        val_d_loss = 0
        val_g_loss = 0
        val_real_labels = []
        val_real_outputs = []
        val_fake_labels = []
        val_fake_outputs = []
        with torch.no_grad():
            for i, (real_samples, labels) in enumerate(val_loader):
                real_samples, labels = real_samples.to(device), labels.to(device)
                batch_size = real_samples.size(0)
                
                real_labels = torch.ones(batch_size, 1).to(device)
                fake_labels = torch.zeros(batch_size, 1).to(device)

                # Validation Discriminator Loss
                outputs = discriminator(real_samples)
                d_loss_real = criterion(outputs[:, 1].view(-1, 1), real_labels)
                noise = torch.randn(batch_size, latent_dim).to(device)
                fake_samples = generator(noise)
                outputs = discriminator(fake_samples)
                d_loss_fake = criterion(outputs[:, 0].view(-1, 1), fake_labels)
                d_loss = d_loss_real + d_loss_fake
                val_d_loss += d_loss.item()
                
                # Validation AUC 계산을 위한 데이터 저장
                val_real_labels.extend(real_labels.cpu().numpy())
                val_real_outputs.extend(outputs[:, 1].view(-1, 1).cpu().numpy())
                val_fake_labels.extend(fake_labels.cpu().numpy())
                val_fake_outputs.extend(outputs[:, 0].view(-1, 1).cpu().numpy())

                # Validation Generator Loss
                noise = torch.randn(batch_size, latent_dim).to(device)
                fake_samples = generator(noise)
                outputs = discriminator(fake_samples)
                g_loss = criterion(outputs[:, 1].view(-1, 1), real_labels)
                val_g_loss += g_loss.item()

            val_d_loss /= len(val_loader)
            val_g_loss /= len(val_loader)
            

            print(f"Epoch [{epoch}/{epochs}] | Train d_loss: {train_d_loss} | Train g_loss: {train_g_loss}")
            print(f"                  | Val d_loss: {val_d_loss} | Val g_loss: {val_g_loss}")


In [ ]:
real_data = np.load('real_mfcc_features.npy')  # numpy array 파일 경로로 대체
fake_data = np.load('fake_mfcc_features.npy')  # numpy array 파일 경로로 대체
unlabeled_data = np.load('unlabeled_mfcc_features.npy')  # numpy array 파일 경로로 대체

In [ ]:
dataset = AudioDataset(real_data, fake_data, unlabeled_data)

In [ ]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [ ]:
latent_dim = 100
generator = Generator(latent_dim)
discriminator = Discriminator()

In [ ]:
train_gan(generator, discriminator, train_loader, val_loader, unlabeled_data, epochs=10000, batch_size=64, latent_dim=latent_dim)